In [1]:
from pyspark.sql.functions import col, udf, lit, when, isnan, monotonically_increasing_id,\
    input_file_name, regexp_extract, to_timestamp, current_timestamp,\
    col, lit, isnan, collect_list, concat_ws, col, max
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType, DataType, TimestampType
from pyspark.sql.types import BooleanType
import re

StatementMeta(, 3fbfdb57-181e-4dfe-b3b3-1b9ffdfb70a7, 3, Finished, Available, Finished, False)

In [2]:
### Email Validation UDF ####
def is_valid_email(val):
    return bool(re.match(r"[^@]+@[^@]+\.[^@]+", val)) if val else False

is_valid_email_udf = udf(is_valid_email, BooleanType())

StatementMeta(, 3fbfdb57-181e-4dfe-b3b3-1b9ffdfb70a7, 4, Finished, Available, Finished, False)

In [3]:
### Load 00 Rules For Delta Table
def load_dq_rules_from_table(config_table,in_table_name,in_tier):
    df = spark.read.format("delta").load(config_table) if "/" in config_table else spark.table(config_table)

    df_rules=df.filter(col("table_name")==in_table_name).filter(col("tier")==in_tier)
    rules = []

    for row in df_rules.filter("active_flag = TRUE").collect():
        rule = {
            "id": row["rule_id"],
            "column": row["column_name"],
            "check": row["check_type"],
            "nullable": row["nullable"],
            "type": row["expected_type"],
            "severity": row["severity"]
        }
        if row["check_type"]== "range":
            rule["range"] = [int(x.strip()) for x in row["check_param"].split(",")]
        elif row["check_type"] in ["format","type","domain"]:
            rule[row["check_type"]] = row["check_param"]
        rules.append(rule)
    return rules

StatementMeta(, 3fbfdb57-181e-4dfe-b3b3-1b9ffdfb70a7, 5, Finished, Available, Finished, False)

In [4]:
def apply_dq_checks_from_rules(df, rules):
    # add row traceability column assumed to already exist in df
    for tc in trance_columns:
        if tc not in df.columns:
            raise ValueError(f"Missing tranceability column: {tc}")

    # delete DQ log schema with traceability
    dq_log_schema = StructType([
        StructField("row_number", LongType(), False),
        StructField("sys_file_timestamp", StringType(), False),
        StructField("sys_insert_timestamp", StringType(), False),
        StructField("source", StringType(), False),
        StructField("rule_id", StringType(), False),
        StructField("column_name", StringType(), False),
        StructField("issue", StringType(), False),
        StructField("check_type", StringType(), False),
        StructField("severity", StringType(), False),
    ])

    dq_log_df = spark.createDataFrame([],dq_log_schema)
    df_checked = df

    for rule in rules:
        col_name = rule["column"]
        rule_id = rule["id"]
        severity = rule["severity"]
        check_type = rule["check"]

        if col_name not in df.columns:
            new_log = [(None,None,None,None,rule_id,col_name,"Column not presant in table", check_type,severity)]
            # dq_log_df = dq_log_df.uniun(spark.createDataFrame(new_log, dq_log_schema))
            continue

        if rule.get("nullable") is False:
            null_cond = col(col_name).isNull() | isnan(col(col_name))
            dq_log_df = dq_log_df.union(
                df.filter(null_cond)
                  .select(*trace_column,
                            lit(rule_id), lit(col_name), lit("Null found"),
                            lit("null"), lit(severity))
            )

        if check_type == "format" and rule.get("format") == "email":
            fail_col = f"{col_name}_format_fail"
            df_checked = df_checked.withColumn(fail_col, ~is_valid_email_udf(col(col_name)))
            dq_log_df = dq_log_df.union(
                df_checked.filter(col(fail_col))
                          .select(*trace_column,
                            lit(rule_id), lit(col_name), lit("Invalid email"),
                            lit("format"), lit(severity))
            )

        if check_type == "range":
            min_val, max_val = rule["range"]
            fail_col = f"{col_name}_range_fail"
            df_checked = df_checked.withColumn(fail_col, (col(column_name)< min_val) | (col(col_name)> max_val))
            dq_log_df = dq_log_df.union(
                df_checked.filter(col(fail_col))
                          .select(*trace_column,
                            lit(rule_id), lit(col_name), lit(f"Out of range [{min_val},{max_val}]"),
                            lit("range"), lit(severity))
            )

        if check_type == "type":
            try:
                fail_col = f"{col_name}_type_fail"
                df_checked = df_checked.withColumn(fail_col, 
                    when(col(col_name).cast(rule["type"]).isNull() & col(col_name).isNotNull(), True).otherwise(False)
                )
                dq_log_df = dq_log_df.union(
                df_checked.filter(col(fail_col))
                          .select(*trace_column,
                            lit(rule_id), lit(col_name), lit(f"Fail to cast to {rule['type']}"),
                            lit("type"), lit(severity))
                            )
            except:
                new_log = [(None,None,None,None,rule_id,col_name, 
                            f"fail to cast to {rule['type']}", severity)]
                dq_log_df = dq_log_df.union(spark.createDataFrame(new_log, dq_log_schema)) 

            if check_type == "domain":
                valid_values = [v.strip() for v in rule["domain"].split(",")]
                fail_col = f"{col_name}_domail_fail"
                df_checked = df_checked.withColumn(fail_col,~col(col_name).isin(valid_values))
                dq_log_df = dq_log_df.union(
                    df_checked.fiter(col(fail_col))
                              .select(*trace_column,
                               lit(rule_id), lit(col_name), 
                               lit("Invalid domain Value"),
                               lit("domain"), lit(severity))
                )

    return dq_log_df.limit(10000)


StatementMeta(, 3fbfdb57-181e-4dfe-b3b3-1b9ffdfb70a7, 6, Finished, Available, Finished, False)

In [5]:
def fn_execute_dq(df,lakehouse,config_schema,config_table,table_name,tier):
    # Execute full DQ flow
    input_config_table=f"{lakehouse}.{config_schema}.{config_table}"

    dq_rules = load_dq_rules_from_table(input_config_table,table_name,tier)
    print("Data read done from dq config")

    log_df = apply_dq_checks_from_rules(df,dq_rules)
    print("DQ failure logs generated")

    # group by row_number and aggregate issue into comma-separeted string
    grouped_log_df = log_df.withColumn("issue_column_name",concat(col("issue"),lit(': '),col("column_name")))\
                            .groupBy("row_number","source")\
                            .agg(concat_ws(", ", collect_list("issue_column_name"))\
                            .alias("issue_summary")
                            )
    print("DQ issue summary generated")

    invalid_df = df.join(grouped_log_df, on=["row_number","Source"], how="inner")\
                    .drop(grouped_log_df.row_number, grouped_log_df.Source) 

    print("Quarantine dataframe generated")

    valid_df = df.join(grouped_log_df, on=["row_number","Source"], how="left_anti")\
                 .drop(grouped_log_df.row_number, grouped_log_df.Source)
    
    print("Filtered dataframe generated")

    return log_df,invalid_df,valid_df

StatementMeta(, 3fbfdb57-181e-4dfe-b3b3-1b9ffdfb70a7, 7, Finished, Available, Finished, False)